# Executando Comandos no Amazon Bedrock AgentCore Code Interpreter - Tutorial

Este tutorial demonstra como usar o Amazon Bedrock AgentCore Code Interpreter para executar comandos (shell e AWS CLI). Iremos interagir com serviços AWS, focando especificamente em operações S3. Vamos percorrer:

1. Criando um interpretador de código
2. Iniciar sessão do interpretador de código
3. Executar Comandos (shell e AWS CLI)
5. Realizar operações S3 (criar bucket, copiar objetos, listar objetos do bucket)
6. Limpeza (parar sessão e excluir interpretador de código)



## Pré-requisitos
- Conta AWS com acesso ao Bedrock AgentCore Code Interpreter
- Você tem as permissões IAM necessárias para criar e gerenciar recursos do interpretador de código
- Você tem as permissões IAM necessárias para realizar operações S3
- Pacotes Python necessários instalados (incluindo boto3 & bedrock-agentcore)


## Seu IAM execution role deve ter a seguinte política IAM anexada

~~~ {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:CreateCodeInterpreter",
                "bedrock-agentcore:StartCodeInterpreterSession",
                "bedrock-agentcore:InvokeCodeInterpreter",
                "bedrock-agentcore:StopCodeInterpreterSession",
                "bedrock-agentcore:DeleteCodeInterpreter",
                "bedrock-agentcore:ListCodeInterpreters",
                "bedrock-agentcore:GetCodeInterpreter"
            ],
            "Resource": "*"
        },
        {
            "Effect": "Allow",
            "Action": [
                "logs:CreateLogGroup",
                "logs:CreateLogStream",
                "logs:PutLogEvents"
            ],
            "Resource": "arn:aws:logs:*:*:log-group:/aws/bedrock-agentcore/code-interpreter*"
        }
    ]
}

#### O IAM execution role também deve ter a seguinte política de confiança anexada. Ao incluir bedrock-agentcore.amazonaws.com na política de confiança, você está permitindo que o próprio serviço Bedrock Agent assuma esta função IAM e realize ações em seu nome (como Amazon S3)

```{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "AWS": "arn:aws:iam::<account_id>:root",
                "Service": [
                    "bedrock-agentcore.amazonaws.com"
                ]
            },
            "Action": "sts:AssumeRole",
            "Condition": {}
        }
    ]
}

Além disso, você precisa anexar a política IAM AmazonS3FullAccess ao IAM Execution role para realizar as operações S3 descritas neste tutorial

## Como funciona

O sandbox de execução de código permite que agentes processem consultas de usuários de forma segura ao criar um ambiente isolado com interpretador de código, shell e sistema de arquivos. Após um Large Language Model auxiliar na seleção de ferramentas, o código é executado dentro desta sessão, antes de ser retornado ao usuário ou agente para síntese.

![architecture local](code-interpreter.png)

## 1. Configurando o Ambiente

Primeiro, vamos importar as bibliotecas necessárias

In [ ]:
!pip install --upgrade -r requirements.txt

In [ ]:
import json
import boto3
from bedrock_agentcore.tools.code_interpreter_client import code_session, CodeInterpreter
from bedrock_agentcore._utils import endpoints
import time
from typing import Dict, Any, List

## 2. Variáveis de Configuração

Configure as variáveis de configuração necessárias para o interpretador de código e operações S3. Também fornecemos uma IAM execution role para passar ao interpretador de código, para que ele possa assumi-la e acessar outros recursos AWS. Esta role precisa de permissões S3, conforme discutido acima

In [ ]:
# Configuração
execution_role_arn = "<execution-role-arn>"
unique_bucket_name = f"amzn-bucket-{int(time.time())}"
s3_path = f"s3://{unique_bucket_name}"

region = "us-west-2"

# arquivo local que vamos fazer upload para o Code interpreter e depois para um bucket S3
local_file = "samples/stats.py"

## 3. Configurando endpoints

Precisamos configurar tanto o endpoint do data plane quanto o control plane para criar os clientes boto3.

In [ ]:
# Configurar endpoints
data_plane_endpoint = endpoints.get_data_plane_endpoint(region)
control_plane_endpoint = endpoints.get_control_plane_endpoint(region)

## 4. Criando Clientes AWS

Inicialize clientes boto3 para operações tanto do control plane quanto do data plane.

In [ ]:
# Criar clientes boto3
cp_client = boto3.client("bedrock-agentcore-control", 
                        region_name=region,
                        endpoint_url=control_plane_endpoint)

dp_client = boto3.client("bedrock-agentcore", 
                        region_name=region,
                        endpoint_url=data_plane_endpoint)

## 5. Criando um Interpretador de Código

Crie uma instância de interpretador de código com parâmetros de configuração específicos.

Ao configurar um interpretador de código, você pode escolher configurações de rede (Sandbox, Public ou VPC), configuração de dependências, configurações de segurança e permissões através de uma IAM runtime role que define quais recursos AWS o interpretador de código pode acessar.

In [ ]:
# Criar interpretador de código
unique_name = f"s3InteractionEnv_{int(time.time())}"
interpreter_response = cp_client.create_code_interpreter(
    name=unique_name,
    description="Ambiente para operações de arquivo S3",
    executionRoleArn=execution_role_arn,
    networkConfiguration={
        'networkMode': 'PUBLIC'
    }
)
interpreter_id = interpreter_response["codeInterpreterId"]
print(f"Interpretador criado: {interpreter_id}")

## 6. Iniciando uma Sessão

Crie uma sessão dentro do interpretador de código para executar código.

In [ ]:
# Iniciar sessão
session_response = dp_client.start_code_interpreter_session(
    codeInterpreterIdentifier=interpreter_id,
    name="s3InteractionSession",
    sessionTimeoutSeconds=900
)
session_id = session_response["sessionId"]
print(f"Sessão criada: {session_id}")

## 7. Função Auxiliar para Execução de Ferramentas

Defina uma função utilitária para simplificar a invocação de ferramentas do interpretador de código

In [ ]:
def call_tool(tool_name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
    response = dp_client.invoke_code_interpreter(
        codeInterpreterIdentifier=interpreter_id,
        sessionId=session_id,
        name=tool_name,
        arguments=arguments
    )
    for event in response["stream"]:
        return json.dumps(event["result"], indent=2)

## 8. Testar Execução de Código

### 8.1 Teste o interpretador de código com um exemplo simples Hello World.

In [ ]:
# Testar execução de código
# Operações S3
print("executando comando shell \n")
command_response = call_tool("executeCommand",
                              {"command": "echo 'Hello World'"})
print(f"resultado do comando: {command_response}")

# Analisar e exibir resultados
command_results = json.loads(command_response)
print(command_results['structuredContent']['stdout'])

### 8.2 A seguir, vamos instalar boto3 usando PIP no sandbox

In [ ]:
# Testar execução de código
# Operações S3
print("executando comando shell \n")
command_response = call_tool("executeCommand",
                              {"command": "pip install boto3"})

# Analisar e exibir resultados
command_results = json.loads(command_response)
print(command_results['structuredContent']['stdout'])

## 9. Operações de Arquivo e Interação com S3 executando comandos

#### 9.1 Escrever arquivo local no sandbox

In [ ]:
# Escrever arquivo no sandbox
print("Escrevendo arquivo no sandbox")
try:
    with open(local_file, 'r', encoding='utf-8') as local_file_content:
        local_file_content = local_file_content.read()
except FileNotFoundError:
    print(f"Erro: O arquivo '{local_file}' não foi encontrado.")
except Exception as e:
    print(f"Ocorreu um erro: {e}")

files_to_create = [{
        "path": "stats.py",
        "text": local_file_content
}]
write_files_response = call_tool("writeFiles", {"content": files_to_create})
print(f"resultado de escrever arquivos: {write_files_response}")

#### 9.2 Criar um Bucket S3 via interpretador de código

In [ ]:
# Operações S3
print("\nCriando bucket S3")
create_s3_response = call_tool("executeCommand",
                              {"command": f"aws s3 mb {s3_path} --region {region}"})
print(f"resultado da criação: {create_s3_response}")

#### 9.3 Fazer upload de arquivo do interpretador de código executando comando para o Bucket S3 (criado acima)

In [ ]:
print("\nFazendo upload de arquivo para S3")
upload_to_s3_response = call_tool("executeCommand",
                                 {"command": f"aws s3 cp {files_to_create[0]['path']} {s3_path}"})
print(f"resultado do upload: {upload_to_s3_response}")



#### 9.4 Listar arquivos do bucket S3 executando um comando no interpretador de código

In [ ]:
print("\nListando arquivos no S3")
list_s3_response = call_tool("executeCommand",
                            {"command": f"aws s3 ls {s3_path}"})
print(f"resultado da listagem: {list_s3_response}")

## 10. Limpeza

Limpe os recursos parando a sessão e excluindo o interpretador.

In [ ]:
# Limpeza
print("Limpando sessão e interpretador")
dp_client.stop_code_interpreter_session(
    codeInterpreterIdentifier=interpreter_id,
    sessionId=session_id
)
print("Sessão parada com sucesso")

cp_client.delete_code_interpreter(codeInterpreterId=interpreter_id)
print("Interpretador excluído com sucesso")